# Parameter recovery demo

This notebook estimates selected physical parameters from a synthetic observation window. The default parameter values are treated as the truth, the first observed state is fixed as `U0`, and Adam updates only the chosen parameters. During optimization it prints progress diagnostics and stores loss, parameter, and gradient-norm histories for plotting.

In [1]:
from pathlib import Path
import sys

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dqgs import (
    DEFAULT_PARAM_VECTOR,
    fit_parameters,
    make_tendency,
    param_indices,
    require_x64,
    simulate,
    theta_to_phi,
    trajectory_loss,
)

require_x64()

## Settings

Start with one parameter. After the one-parameter case works, edit `FREE_NAMES` to try two or three parameters.

In [2]:
FREE_NAMES = ("k_d",)
DT = 0.1
N_STEPS = 100
WRITE_STEPS = 1
INITIAL_GUESS_MULTIPLIER = 1.05
LEARNING_RATE = 1e-2
NUM_ITERATIONS = 300
PROGRESS_EVERY = 1

IC_PATH = ROOT / "configs" / "ic" / "de_cruz_active_ic.npy"

## Generate synthetic observations

`obs` is the fixed fake observation window. It is generated once from the validated default parameters.

In [3]:
f_theta = make_tendency(FREE_NAMES)
theta_ref = jnp.asarray(DEFAULT_PARAM_VECTOR[list(param_indices(FREE_NAMES))])

theta_true = theta_ref
theta0 = theta_ref * INITIAL_GUESS_MULTIPLIER

initial_U0 = jnp.asarray(np.load(IC_PATH))
obs = simulate(f_theta, theta_true, initial_U0, DT, N_STEPS, WRITE_STEPS)

print("free parameters:", FREE_NAMES)
print("obs shape:", obs.shape)
print("theta_true:", np.asarray(theta_true))
print("theta0:", np.asarray(theta0))

free parameters: ('k_d',)
obs shape: (101, 36)
theta_true: [0.029]
theta0: [0.03045]


## Fit parameters

The loss fixes `U0 = obs[0]`, integrates a predicted trajectory from the current guessed parameters, and compares only future states: `mean((pred[1:] - obs[1:]) ** 2)`. The optimizer prints diagnostics every `PROGRESS_EVERY` iterations and stores the curves in `result.loss_history`, `result.theta_history`, and `result.grad_norm_history`.

In [4]:
initial_phi = theta_to_phi(theta0, theta_ref)
initial_loss = trajectory_loss(
    initial_phi, theta_ref, f_theta, obs, DT, N_STEPS, WRITE_STEPS
)

result = fit_parameters(
    f_theta=f_theta,
    theta0=theta0,
    theta_ref=theta_ref,
    obs=obs,
    dt=DT,
    n_steps=N_STEPS,
    write_steps=WRITE_STEPS,
    learning_rate=LEARNING_RATE,
    num_iterations=NUM_ITERATIONS,
    progress_every=PROGRESS_EVERY,
    parameter_names=FREE_NAMES,
)

final_loss = trajectory_loss(
    result.phi, theta_ref, f_theta, obs, DT, N_STEPS, WRITE_STEPS
)

print(f"initial loss: {float(initial_loss):.16e}")
print(f"final loss:   {float(final_loss):.16e}")

iter     1/300: loss=8.470979e-10, grad_norm=3.545140e-08, k_d=0.030213418
iter     2/300: loss=5.939141e-10, grad_norm=2.948805e-08, k_d=0.029985138
iter     3/300: loss=3.919075e-10, grad_norm=2.379961e-08, k_d=0.029767017
iter     4/300: loss=2.378292e-10, grad_norm=1.842493e-08, k_d=0.029561121
iter     5/300: loss=1.274116e-10, grad_norm=1.340612e-08, k_d=0.029369663
iter     6/300: loss=5.534974e-11, grad_norm=8.787064e-09, k_d=0.029194884
iter     7/300: loss=1.539687e-11, grad_norm=4.610878e-09, k_d=0.029038897
iter     8/300: loss=6.138112e-13, grad_norm=9.164138e-10, k_d=0.028903471
iter     9/300: loss=3.782812e-12, grad_norm=2.265902e-09, k_d=0.028789842
iter    10/300: loss=1.794059e-11, grad_norm=4.917953e-09, k_d=0.028698556
iter    11/300: loss=3.692766e-11, grad_norm=7.036519e-09, k_d=0.028629425
iter    12/300: loss=5.582639e-11, grad_norm=8.633804e-09, k_d=0.028581571
iter    13/300: loss=7.119227e-11, grad_norm=9.735868e-09, k_d=0.028553558
iter    14/300: loss=8.10

## Inspect recovery

In [ ]:
for name, truth, guess, recovered in zip(
    FREE_NAMES,
    np.asarray(theta_true),
    np.asarray(theta0),
    np.asarray(result.theta),
    strict=True,
):
    print(f"{name}:")
    print(f"  true/default: {truth:.16g}")
    print(f"  initial:      {guess:.16g}")
    print(f"  recovered:    {recovered:.16g}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(result.loss_history)
axes[0].set_xlabel("Adam iteration")
axes[0].set_ylabel("trajectory MSE")
axes[0].set_title("Loss history")

for i, name in enumerate(FREE_NAMES):
    axes[1].plot(result.theta_history[:, i], label=name)
    axes[1].axhline(float(theta_true[i]), linestyle="--", linewidth=1)
axes[1].set_xlabel("Adam iteration")
axes[1].set_ylabel("parameter value")
axes[1].set_title("Parameter history")
axes[1].legend()

axes[2].semilogy(result.grad_norm_history)
axes[2].set_xlabel("Adam iteration")
axes[2].set_ylabel("gradient norm")
axes[2].set_title("Gradient norm")

fig.tight_layout()

In [ ]:
time = DT * WRITE_STEPS * np.arange(obs.shape[0])
state_index = 0

plt.figure(figsize=(7, 4))
plt.plot(time, np.asarray(obs[:, state_index]), label="obs")
plt.plot(time, np.asarray(result.predicted[:, state_index]), "--", label="pred")
plt.xlabel("time")
plt.ylabel(f"U[{state_index}]")
plt.title("Observed vs recovered trajectory")
plt.legend()
plt.tight_layout()